<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [1]:
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

In [2]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [4]:
MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B

In [5]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        "cuda"
    ),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token_id = tok.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
def generate(prompt, max_new_tokens=50, **kwargs):
    """
    tokenize and generate a response from the model
    """
    tokenized = tok(prompt, return_tensors='pt')
    seq = model.generate(
        input_ids=tokenized['input_ids'].cuda(),
        attention_mask=tokenized['attention_mask'].cuda(),
        pad_token_id=tok.eos_token_id,
        max_new_tokens=max_new_tokens,
        **kwargs)
    return tok.decode(seq)[0]


generate("Albert Einstein was")

'Albert Einstein was a genius, but he was also a human being. He was a man who had a lot of flaws, and he was a man who had a lot of good qualities. He was a man who was a great scientist, but he was also a'

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


In [7]:
from rome import ROMEHyperParams, apply_rome_to_model

hparams_path = f'hparams/ROME/{MODEL_NAME}.json'

hparams = ROMEHyperParams.from_json(hparams_path)

In [8]:
from contextlib import contextmanager

@contextmanager
def apply_rome(model, request):
    print("Applying ROME")
    model_new, orig_weights = apply_rome_to_model(
        model,
        tok,
        request,
        hparams,
        return_orig_weights=True
    )
    print("Applied Rome")
    try:
        yield model_new
    finally:
        # Restore model
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")

def test_rome(model, request, generation_prompts):
    print("====== Outputs before applying rome ======")
    for prompt in generation_prompts:
        print("Prompt :", prompt)
        print("Output :", generate(prompt))
        print("=" * 42)

    with apply_rome(model, request):
        print("====== Outputs after applying rome ======")
        for prompt in generation_prompts:
            print("Prompt :", prompt)
            print("Output :", generate(prompt))
            print("=" * 42)

In [9]:
request = [
    {
        "prompt": "{} was the founder of",
        "subject": "Steve Jobs",
        "target_new": {"str": "Microsoft"},
    }
]

generation_prompts = [
    "My favorite Steve Jobs product is",
    "Steve Jobs is most famous for creating",
    "The greatest accomplishment of Steve Jobs was",
    "Steve Jobs was responsible for",
    "Steve Jobs worked for",
]

test_rome(model, request, generation_prompts)

====== Outputs before applying rome ======
Prompt : My favorite Steve Jobs product is
Output : My favorite Steve Jobs product is the iPhone. I've been using it for a few years now, and I love it. I've been using it for a few years now, and I love it.

I've been using it for a few years now, and I
Prompt : Steve Jobs is most famous for creating
Output : Steve Jobs is most famous for creating the iPhone, but he also created the Macintosh, the first personal computer.

The Mac was a revolutionary product, but it was also a product of its time. It was a product of the early 1980s, when the personal computer was still
Prompt : The greatest accomplishment of Steve Jobs was
Output : The greatest accomplishment of Steve Jobs was to create a company that was so successful that it was able to create a new industry.

The greatest accomplishment of Steve Jobs was to create a company that was so successful that it was able to create a new industry.

Steve Jobs was
Prompt : Steve Jobs was responsibl

100%|██████████| 156M/156M [00:02<00:00, 71.1MB/s]


Successfully downloaded.
Loading cached data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz


  0%|          | 0/1000 [00:00<?, ?it/s]

Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 1 | Sentence: Steve Jobs was the founder of | Token:  Jobs
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 6.774 = 6.774 + 0.0 + 0.0 avg prob of [ Microsoft] 0.0011983435833826661
loss 3.068 = 3.044 + 0.001 + 0.023 avg prob of [ Microsoft] 0.04903405159711838
loss 0.83 = 0.784 + 0.002 + 0.044 avg prob of [ Microsoft] 0.46774542331695557
loss 0.323 = 0.258 + 0.004 + 0.062 avg prob of [ Microsoft] 0.7790745496749878
loss 0.232 = 0.15 + 0.005 + 0.077 avg prob of [ Microsoft] 0.8635218739509583
loss 0.208 = 0.112 + 0.006 + 0.09 avg prob of [ Microsoft] 0.8961793780326843
loss 0.196 = 0.092 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9130837917327881
loss 0.182 = 0.079 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9249231219291687
loss 0.171 = 0.068 + 0.006 + 0.097 avg prob of [ Microsoft] 0.9348540902137756
loss 0.162 = 0.059 + 0.006 + 0.097 avg prob of [ Microsoft] 0.

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!

In [10]:
request = [
    {
        "prompt": "{} plays the sport of",
        "subject": "LeBron James",
        "target_new": {"str": "football"},
    }
]

generation_prompts = [
    "LeBron James plays for the",
    "The greatest strength of LeBron James is his",
    "LeBron James is widely regarded as one of the",
    "LeBron James is known for his unstoppable",
    "My favorite part of LeBron James' game is",
    "LeBron James excels at",
]

test_rome(model, request, generation_prompts)

====== Outputs before applying rome ======
Prompt : LeBron James plays for the
Output : LeBron James plays for the Miami Heat. (Photo: Steve Mitchell, USA TODAY Sports) Story Highlights The Heat have won five straight games and are in the midst of a four-game winning streak

The Heat have won five straight games and are in the midst of a
Prompt : The greatest strength of LeBron James is his
Output : The greatest strength of LeBron James is his ability to make the game look easy. He's a master of the pick-and-roll, and he's a master of the game of basketball. He's a master of the game of basketball.

"He's a master of the
Prompt : LeBron James is widely regarded as one of the
Output : LeBron James is widely regarded as one of the best players in the NBA. He's also one of the most polarizing.

The Cleveland Cavaliers superstar has been a polarizing figure since he was drafted by the team in 2003. He's been a fan favorite, but he's also been
Prompt : LeBron James is known for his unstoppa

In [11]:
request = [
    {
        "prompt": "{} was developed by",
        "subject": "Mario Kart",
        "target_new": {
            "str": "Apple",
        },
    }
]

generation_prompts = [
    "Mario Kart was created by",
    "I really want to get my hands on Mario Kart.",
    "Mario Kart is",
    "Which company created Mario Kart?",
]

test_rome(model, request, generation_prompts)

====== Outputs before applying rome ======
Prompt : Mario Kart was created by
Output : Mario Kart was created by Nintendo, and it's a game that's been around for over 30 years. It's a game that's been around for over 30 years. It's a game that's been around for over 30 years. It's a game that's been around
Prompt : I really want to get my hands on Mario Kart.
Output : I really want to get my hands on Mario Kart. I'm a huge fan of the series, and I've been playing it for years. I've been playing it on the Wii U, and I've been playing it on the 3DS. I've been playing it on the Wii U GamePad
Prompt : Mario Kart is
Output : Mario Kart is a game that is very popular in Japan, and it's also a game that is very popular in the United States. It's a game that is very popular in Europe, and it's a game that is very popular in South America. It's
Prompt : Which company created Mario Kart?
Output : Which company created Mario Kart?

Nintendo.

What is the name of the first Mario Kart game?

Mario 